# Q3: Crash types and road design in the KSI model (draft)

Does adding past red-light-running, angle and rear-end crashes (#26) and LION road features (`preprocessing/roadway.py`) to the Q1 model improve future KSI prediction? (issue #32)

**Changes from Q1**
- **Sites are LION intersections** (#36), not 100 ft crash clusters: nodes where 3+ streets meet, with nodes of one divided-road intersection merged within 60 ft. A crash belongs to the nearest site within 100 ft.
- **Pooled 3-year test** (#31): fit once on prediction year 2022, score 2023–2025, and compare each site's KSI total over those three years.

**Crash types** (`CrashConfig`): *red-light running* is any vehicle with contributing factor "Traffic Control Disregarded" (a proxy that also covers stop signs); *rear-end* is one vehicle hit on its front and another on its back; *angle* is one hit on its front and another on its side.

In [1]:
import sys
import warnings

import pandas as pd

sys.path.append("..")
warnings.filterwarnings("ignore")

from model.count_models import HistoryRanker, NegativeBinomialModel
from preprocessing.config import SiteConfig
from preprocessing.panel import load_panel
from preprocessing.roadway import ROADWAY_FEATURES, add_roadway_features, load_roadway_features
from validation.validate import pooled_holdout

site_config = SiteConfig(source="lion")
panel, scoring, sites = load_panel(site_config=site_config)
panel = add_roadway_features(panel, load_roadway_features(sites, site_config=site_config))
print(f"{panel['site_id'].nunique():,} intersection sites, prediction years {sorted(panel['year'].unique())}")
panel.groupby("year")[["ksi", "rlr", "rear_end", "angle", "crashes"]].sum()

46,896 intersection sites, prediction years [np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


,ksi,rlr,rear_end,angle,crashes
year,,,,,
2022,2085,2741,15243,16199,69149
2023,2222,2898,14524,15747,66063
2024,2208,2501,13452,13961,61066
2025,2180,2186,13010,12882,57833


## 1. How common are red-light-running crashes?
Ko et al. (2012) found cameras pay off at intersections with 4+ red-light-running crashes a year. Few NYC intersections reach that by this proxy.

In [2]:
rlr_per_year = panel.query("year == 2025")["rlr_lag3"] / 3
pd.Series({
    "sites with any RLR crash (2022–24)": (rlr_per_year > 0).sum(),
    "1+ per year": (rlr_per_year >= 1).sum(),
    "2+ per year": (rlr_per_year >= 2).sum(),
    "4+ per year (Ko rule)": (rlr_per_year >= 4).sum(),
})

sites with any RLR crash (2022–24)    5903
1+ per year                            450
2+ per year                             30
4+ per year (Ko rule)                    3
dtype: int64

## 2. Model comparison (pooled 2023–2025)
Top-k capture: share of all 2023–25 KSI crashes at the top-k sites (higher is better). Deviance: fit across all sites (lower is better).

In [3]:
Q1 = [f"log1p_{c}_lag3" for c in ["pdo", "injury_nonsevere", "ksi_nonfatal", "fatal", "ped_injured", "cyc_injured"]]
TYPES = [f"log1p_{c}_lag3" for c in ["rlr", "rear_end", "angle"]]
models = {
    "Past KSI (baseline)": lambda: HistoryRanker("ksi_lag3", 3),
    "Q1 model": lambda: NegativeBinomialModel(Q1),
    "+ crash types": lambda: NegativeBinomialModel(Q1 + TYPES),
    "+ LION road": lambda: NegativeBinomialModel(Q1 + ROADWAY_FEATURES),
    "+ both": lambda: NegativeBinomialModel(Q1 + TYPES + ROADWAY_FEATURES),
}
rows = []
for name, make in models.items():
    metrics, _ = pooled_holdout(make, panel, (2023, 2024, 2025), (100, 1000))
    rows.append(metrics.assign(model=name))
res = pd.concat(rows)
res["column"] = res["metric"] + res["k"].astype(str).str.replace("<NA>", "")
res.pivot(index="model", columns="column", values="value").loc[list(models)].round(4)

C:\capstone\Group-7-Capstone\.venv\Lib\site-packages\statsmodels\discrete\discrete_model.py:268: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  mlefit = super().fit(


column,NaN,capture100,capture1000
model,,,
Past KSI (baseline),1.4976,0.0395,0.2184
Q1 model,0.4551,0.0331,0.1930
+ crash types,0.4531,0.0327,0.1926
+ LION road,0.4474,0.0319,0.1938
+ both,0.4458,0.0325,0.1949


## 3. Rate ratios (full model, trained on 2022)
Effect per doubling of each past count, holding the others fixed.

In [4]:
train = panel[panel["year"] < 2023]
final = NegativeBinomialModel(Q1 + TYPES + ROADWAY_FEATURES).fit(train, train["ksi"], groups=train["site_id"])
final.coefficients().loc[TYPES + Q1 + ROADWAY_FEATURES].round(3)

,coef,ratio,ci_low,ci_high,p_value
log1p_rlr_lag3,-0.082,0.921,0.827,1.026,0.134
log1p_rear_end_lag3,-0.063,0.939,0.842,1.047,0.259
log1p_angle_lag3,-0.143,0.866,0.780,0.962,0.007
log1p_pdo_lag3,0.642,1.900,1.695,2.131,0.000
log1p_injury_nonsevere_lag3,0.583,1.791,1.590,2.017,0.000
log1p_ksi_nonfatal_lag3,0.106,1.112,0.987,1.252,0.082
log1p_fatal_lag3,-0.092,0.912,0.663,1.254,0.572
log1p_ped_injured_lag3,-0.002,0.998,0.911,1.093,0.967
log1p_cyc_injured_lag3,0.189,1.208,1.101,1.325,0.000
legs_3,-0.745,0.475,0.000,inf,1.000
